In [ ]:
# x-axis: dose of morphine, y-axis: inferred conversion coefficient
# x-axis: dose or morphine, y-axis: ratio of recommended opioid to Nielson recommendation

# DATAFRAME:
    ## PROMPT includes NIELSON conversion coefficient
    ## PROMPT includes EXAMPLES
    ## ADMINISTRATION_METHOD
    ## SUBSTANCE
    ## DOSE_FROM (random based on 10 mg to 120 mg oral morphine)
    ## EXPECTED DOSE_TO
    ## OBSERVED DOSE_TO FROM LLM
    ## RATIO OBSERVED/EXPECTED

# AGGREGATES:
    ## PROMPT includes NIELSON conversion coefficient
    ## PROMPT includes EXAMPLES
    ## ADMINISTRATION_METHOD
    ## SUBSTANCE
    ## PROPORTION of RATIO outside of 0.7 - 1.3

    # p-value for a non-zero slope
    # p-value for a non-one mean

    # CDF of RATIO EXPECTED/OBSERVED by including Nielson, including examples

In [ ]:
# helper function
# generate a random dose of opioid based on OME
# choose a random value on a log scale between 10 mg and 120 mg
# use Neilson 2015 to convert from OME to DOSE_FROM

In [ ]:
# prompt function
# generate a prompt from inputs:
    # INCLUDE NIELSON?
    # INCLUDE 10 mg and 120 mg EXAMPLES?
    # ADMINISTRATION_METHOD

In [ ]:
# an OPIOID_FROM and an OPIOID_TO both have a ADMINISTRATION_METHOD and SUBSTANCE
# ask to convert from DOSE_FROM of OPIOID_FROM to OPIOID_TO
# include a conversion coefficient indicating the oral morphine equivalent

In [ ]:
# Create a prompted_conversion object equivalent to a row of the dataframe
# has properties:
                
    ## ADMINISTRATION_METHOD_FROM
    ## SUBSTANCE_FROM
        ## DOSE_FROM (random based on 10 mg to 120 mg oral morphine)

    ## ADMINISTRATION_METHOD_TO (default oral)
    ## SUBSTANCE_TO (default morphine)
        ## EXPECTED DOSE_TO ACCORDING TO NIELSON
            ## PROMPT includes NIELSON conversion coefficient
            ## PROMPT includes EXAMPLES
                # PROMPT
                # MODEL PROMPTED
                    # RESPONSE
                    # OBSERVED DOSE_TO
                    # RATIO OBSERVED/EXPECTED
                    # RATIO INSIDE 0.7 - 1.3

In [ ]:
# Create an experiment object
    # INPUT:
        # TUPLES of (FROM, TO) or a LIST of FROM, defaulting to morphine
        # NUMBER of REPETITIONS
        # INCLUDE NIELSON?
        # INCLUDE 10 mg and 120 mg EXAMPLES?
    # OUTPUT:
        # LIST of prompted_conversions
        # DATAFRAME of prompted_conversions
        # AGGREGATES
            # by (FROM, TO) and total
                # Regression of EXPECTED DOSE vs. RATIO OBSERVED/EXPECTED
                # p-value non-zero slope
                # p-value non-one mean
            # by (FROM, TO) and total
                # CDF of RATIO OBSERVED/EXPECTED
            # by (FROM, TO) and total
                # Proportion RATIO INSIDE 0.7 - 1.3            
        # FIGURES
            # Scatter plot of EXPECTED DOSE v. RATIO OBSERVED/EXPECTED by (FROM, TO)
                # Should we include a regression line?
            # CDF of RATIO OBSERVED/EXPECTED by (FROM, TO)
            # Proportion of RATIO INSIDE 0.7 - 1.3 by (FROM, TO)



In [1]:
# Conversion Service
from opioid import Opioid
from conversion_service import ConversionService
from nielson_data import NIELSON_CONVERSIONS

conversion_service = ConversionService(NIELSON_CONVERSIONS)

iv_fentanyl = Opioid(administration_method="iv", substance="fentanyl")
oral_hydromorphone = Opioid(administration_method="oral", substance="hydromorphone")
oral_morphine = Opioid(administration_method="oral", substance="morphine")

random_dose = conversion_service.generate_random_dose(iv_fentanyl)
print(f"Random dose for {iv_fentanyl.administration_method}_{iv_fentanyl.substance}: {random_dose:.2f} {iv_fentanyl.unit}")

equivalent_dose_oral_morphine = conversion_service.calculate_expected_dose(random_dose, iv_fentanyl, oral_morphine)
print(f"Equivalent dose for {oral_morphine.administration_method}_{oral_morphine.substance}: {equivalent_dose_oral_morphine:.2f} {oral_morphine.unit}")

Random dose for iv_fentanyl: 0.11 mcg/hour
Equivalent dose for oral_morphine: 11.04 mg/day


In [2]:
print(iv_fentanyl)

Opioid(administration_method='iv', substance='fentanyl', unit='mcg/hour')


In [1]:
# Prompt Service
from prompt_service import PromptService, PromptedOpioidConversion
from conversion_service import ConversionService
from nielson_data import NIELSON_CONVERSIONS
from opioid import Opioid

# Initialize the PromptService
prompt_service = PromptService(ConversionService(NIELSON_CONVERSIONS))

# Define opioids
iv_fentanyl = Opioid(administration_method="iv", substance="fentanyl", unit="mcg/hour")
oral_hydromorphone = Opioid(administration_method="oral", substance="hydromorphone", unit="mg/day")

# Create a PromptedOpioidConversion object without specifying `from_dose` or `conversion_service`
prompted_conversion = PromptedOpioidConversion(
    from_opioid=iv_fentanyl,
    model="gpt-4.1-mini",)

# Generate the prompt using the PromptService instance
generated_prompt = prompted_conversion.generate_prompt(prompt_service)

# Print the generated prompt
print("Generated Prompt:")
print(generated_prompt)

# Print the randomly generated `from_dose` and calculated `expected_to_dose`
print(f"\nRandomly Generated Dose: {prompted_conversion.from_dose:.2f} {iv_fentanyl.unit}")
print(f"Expected Dose: {prompted_conversion.expected_to_dose:.2f} {oral_hydromorphone.unit}")

Generated Prompt:
Convert 0.13 mcg/hour of fentanyl (iv) to mg/day of morphine (oral).
How many mg/day of morphine (oral) is equivalent to 0.13 mcg/hour of fentanyl (iv)?

Use the following conversion factor(s):
100 OME: 1 mcg/hour of fentanyl (iv) is equivalent to 100 mg/day morphine (oral).
100 OME for fentanyl (iv) means 10 mg/day of morphine (oral) is equivalent to 0.10 mcg/hour of fentanyl (iv).
And 120 mg/day of morphine (oral) is equivalent to 1.20 mcg/hour of fentanyl (iv).

Express the response as a single numeric value without any additional text in the format {calculated dose: value}.

Randomly Generated Dose: 0.13 mcg/hour
Expected Dose: 13.00 mg/day


In [5]:
from conversion_service import ConversionService
from prompt_service import PromptService, PromptedOpioidConversion
from nielson_data import NIELSON_CONVERSIONS
from opioid import Opioid

# Initialize the OpenAI client once
PromptedOpioidConversion.initialize_client()

# Define opioids
iv_fentanyl = Opioid(administration_method="iv", substance="fentanyl", unit="mcg/hour")
oral_hydromorphone = Opioid(administration_method="oral", substance="hydromorphone", unit="mg/day")

# Create a PromptedOpioidConversion object
conversion = PromptedOpioidConversion(
    from_opioid=iv_fentanyl,
    include_nielson=True,
    include_examples=False
)

# Generate the prompt
prompt_service=PromptService(ConversionService(NIELSON_CONVERSIONS))
conversion.generate_prompt(prompt_service=prompt_service)

# Query the model
observed_dose = conversion.ask_model()

# Print the results
print("Generated Prompt:")
print(conversion.prompt)
print(f"Observed Dose: {observed_dose:.2f} {oral_hydromorphone.unit}")
print(f"Expected Dose: {conversion.expected_to_dose:.2f} {oral_hydromorphone.unit}")
print(f"Ratio Observed/Expected: {conversion.ratio_observed_expected:.2f}")

Generated Prompt:
Convert 0.32 mcg/hour of fentanyl (iv) to mg/day of morphine (oral).
How many mg/day of morphine (oral) is equivalent to 0.32 mcg/hour of fentanyl (iv)?

Use the following conversion factor(s):
100 OME: 1 mcg/hour of fentanyl (iv) is equivalent to 100 mg/day morphine (oral).

Express the response in the format {'calculated dose': value}.  Do not include any additional text in your response, only the dictionary.
Observed Dose: 32.00 mg/day
Expected Dose: 32.00 mg/day
Ratio Observed/Expected: 1.00
